In [6]:
import os
import random
os.environ['SPCONV_ALGO'] = 'native'        # Can be 'native' or 'auto', default is 'auto'.
                                            # 'auto' is faster but will do benchmarking at the beginning.
                                            # Recommended to set to 'native' if run only once.

import imageio
from PIL import Image
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import render_utils, postprocessing_utils

MODEL_NAME = "lamp_post"
TEXTURE_SIZE = 2048
SIMPLIFY_RATIO = 0.8

In [7]:


# Load a pipeline from pre-trained HuggingFace model
pipeline = TrellisImageTo3DPipeline.from_pretrained("microsoft/TRELLIS-image-large")
pipeline.cuda()

# Load an image
image = Image.open(f"assets/dissertation_assets/{MODEL_NAME}.png")

# Run the pipeline
outputs = pipeline.run(
    image,
    seed=random.randint(1, 9999),
)

Using cache found in /home/liams/.cache/torch/hub/facebookresearch_dinov2_main
Sampling: 100%|██████████| 25/25 [00:05<00:00,  4.83it/s]


In [8]:
# Render the outputs
video = render_utils.render_video(outputs['gaussian'][0])['color']
imageio.mimsave(f"assets/generated_videos/{MODEL_NAME}_gs.mp4", video, fps=30)
video = render_utils.render_video(outputs['radiance_field'][0])['color']
imageio.mimsave(f"assets/generated_videos/{MODEL_NAME}_rf.mp4", video, fps=30)
video = render_utils.render_video(outputs['mesh'][0])['normal']
imageio.mimsave(f"assets/generated_videos/{MODEL_NAME}_mesh.mp4", video, fps=30)

Rendering: 300it [00:01, 193.96it/s]
Rendering: 300it [00:01, 206.16it/s]
Rendering: 300it [00:07, 41.59it/s]


In [9]:
# GLB files can be extracted from the outputs
glb = postprocessing_utils.to_glb(
    outputs['gaussian'][0],
    outputs['mesh'][0],
    # Optional parameters
    simplify=SIMPLIFY_RATIO,          # Ratio of triangles to remove in the simplification process
    texture_size=TEXTURE_SIZE,      # Size of the texture used for the GLB
)
os.makedirs(f"assets/generated_models/{MODEL_NAME}", exist_ok=True)
glb.export(f"assets/generated_models/{MODEL_NAME}/{MODEL_NAME}.glb")

# Save Gaussians as PLY files
os.makedirs(f"assets/generated_plys/{MODEL_NAME}", exist_ok=True)
outputs['gaussian'][0].save_ply(f"assets/generated_plys/{MODEL_NAME}/{MODEL_NAME}.ply")

Before postprocess: 58096 vertices, 116178 faces


Decimating Mesh: 100%|██████████[00:00<00:00]


After decimate: 11622 vertices, 23235 faces


Rasterizing: 100%|██████████| 1000/1000 [00:04<00:00, 201.90it/s]


Found 2556 invisible faces
Dual graph: 34842 edges
Mincut solved, start checking the cut
Removed 2568 faces by mincut

Loading ..96%

100% done 
After remove invisible faces: 10343 vertices, 20685 faces


WARNING- Some cuts were necessary to cope with non manifold configuration.
Rendering: 100it [00:00, 109.08it/s]
Texture baking (opt): optimizing: 100%|██████████| 2500/2500 [08:07<00:00,  5.13it/s, loss=0.0204] 


In [10]:
from IPython.display import Video
import os

# Display the generated videos
print("Gaussian Splatting:")
if os.path.exists(f"assets/generated_videos/{MODEL_NAME}_gs.mp4"):
    display(Video(f"assets/generated_videos/{MODEL_NAME}_gs.mp4", embed=True))

print("\nRadiance Field:")
if os.path.exists(f"assets/generated_videos/{MODEL_NAME}_rf.mp4"):
    display(Video(f"assets/generated_videos/{MODEL_NAME}_rf.mp4", embed=True))

print("\nMesh:")
if os.path.exists(f"assets/generated_videos/{MODEL_NAME}_mesh.mp4"):
    display(Video(f"assets/generated_videos/{MODEL_NAME}_mesh.mp4", embed=True))

Gaussian Splatting:



Radiance Field:



Mesh:
